# Agent-Driven Retrieval [Step 1 - Agent Decides What and When to Retrieve]

> **MLCourse - Agentic AI - Agentic RAG**

In traditional RAG, every query triggers retrieval. In agentic RAG, the
agent decides whether retrieval is needed at all. This notebook builds
a LangGraph agent that queries a vector store only when the LLM determines
it lacks sufficient knowledge to answer. We trace the full decision loop:
reason -> retrieve (maybe) -> reason -> answer.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

[GREEN] No API key needed -- using local ChatOllama


### 1. Load and Chunk the Source Document


In [ ]:
# We use alice.txt as our knowledge base. Split it into manageable chunks
# for embedding and retrieval.

from langchain_text_splitters import CharacterTextSplitter

text_path = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)
print(f"Loaded alice.txt: {len(raw_text)} chars -> {len(chunks)} chunks")
print(f"Sample chunk (first 200 chars): {chunks[0][:200]}...")


### 2. Build the Vector Store


In [ ]:
# Embed all chunks into Chroma for similarity search.

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="alice_rag")
print(f"Vector store built with {vectorstore._collection.count()} vectors")


### 3. Create the Retriever


In [ ]:
# Standard similarity retriever with a k=4 default.

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready (k=4)")


### 4. Initialize the LLM


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("LLM initialized:", llm.model)


### 5. Define the Agent State


In [ ]:
# The state tracks the user query, the agent's decision, retrieved docs,
# and the final answer. The `decision` field is the key: it tells us
# whether the agent chose to retrieve or answer directly.

from typing import TypedDict, Literal
from langchain_core.documents import Document

class RetrievalState(TypedDict):
    query: str
    decision: str           # "retrieve" or "answer_directly"
    retrieved_docs: list    # list of Document objects
    context: str            # concatenated doc content for prompting
    answer: str

print("State defined: query, decision, retrieved_docs, context, answer")


### 6. Build the Decision Node


In [ ]:
# The LLM reads the query and decides whether it needs external knowledge.
# This is the core of agent-driven retrieval: the agent reasons about
# whether it can answer from parametric memory alone.

from langchain_core.prompts import ChatPromptTemplate

decision_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a retrieval decision maker. Given a user query, decide whether "
     "you need to retrieve external documents to answer it well.\n\n"
     "RULES:\n"
     "- Answer 'retrieve' if the query is about specific content in a document "
     "(e.g., a book, article, or dataset) that you cannot reliably answer from memory.\n"
     "- Answer 'answer_directly' if the query is general knowledge you already know, "
     "or if it is a greeting, opinion request, or clarification.\n\n"
     "Reply with ONLY one word: 'retrieve' or 'answer_directly'."),
    ("user", "{query}")
])

def decide_retrieval(state: RetrievalState) -> dict:
    """Node: LLM decides whether to retrieve documents."""
    query = state["query"]
    response = (decision_prompt | llm).invoke({"query": query})
    decision = response.content.strip().lower()
    # Safety fallback
    if decision not in ("retrieve", "answer_directly"):
        decision = "retrieve"
    print(f"  [DECIDE] '{query[:60]}...' -> {decision}")
    return {"decision": decision}


### 7. Build the Retrieval Node


In [ ]:
# Fetches relevant documents from the vector store and concatenates
# their content into a single context string.

def retrieve_docs(state: RetrievalState) -> dict:
    """Node: Retrieve relevant documents from the vector store."""
    query = state["query"]
    docs = retriever.invoke(query)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    print(f"  [RETRIEVE] Got {len(docs)} documents ({len(context)} chars)")
    return {"retrieved_docs": docs, "context": context}


### 8. Build the Answer Nodes


In [ ]:
# Two answer paths: one with retrieved context, one without.

answer_with_context_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's question using ONLY the provided context. "
     "If the context does not contain enough information, say so. "
     "Be concise and accurate."),
    ("user", "Context:\n{context}\n\nQuestion: {query}")
])

answer_direct_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's question from your general knowledge. "
     "Be concise and helpful."),
    ("user", "{query}")
])

def answer_with_retrieval(state: RetrievalState) -> dict:
    """Node: Answer using retrieved context."""
    response = (answer_with_context_prompt | llm).invoke({
        "context": state["context"],
        "query": state["query"]
    })
    print(f"  [ANSWER-WITH-DOCS] {response.content[:80]}...")
    return {"answer": response.content}

def answer_without_retrieval(state: RetrievalState) -> dict:
    """Node: Answer directly (no retrieval)."""
    response = (answer_direct_prompt | llm).invoke({"query": state["query"]})
    print(f"  [ANSWER-DIRECT] {response.content[:80]}...")
    return {"answer": response.content}


### 9. Define the Routing Logic


In [ ]:
# After the decision node, route to retrieval or direct answer.

def route_decision(state: RetrievalState) -> Literal["retrieve_docs", "answer_without_retrieval"]:
    """Route based on the agent's decision."""
    if state["decision"] == "retrieve":
        return "retrieve_docs"
    return "answer_without_retrieval"


### 10. Build the LangGraph


In [ ]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(RetrievalState)

# Nodes
graph_builder.add_node("decide", decide_retrieval)
graph_builder.add_node("retrieve_docs", retrieve_docs)
graph_builder.add_node("answer_with_retrieval", answer_with_retrieval)
graph_builder.add_node("answer_without_retrieval", answer_without_retrieval)

# Edges
graph_builder.add_edge(START, "decide")
graph_builder.add_conditional_edges(
    "decide",
    route_decision,
    {
        "retrieve_docs": "retrieve_docs",
        "answer_without_retrieval": "answer_without_retrieval",
    }
)
graph_builder.add_edge("retrieve_docs", "answer_with_retrieval")
graph_builder.add_edge("answer_with_retrieval", END)
graph_builder.add_edge("answer_without_retrieval", END)

graph = graph_builder.compile()
print("Agent-driven retrieval graph compiled.")


### 11. Visualize the Graph


In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not render graph image: {e}")
    print("Graph structure:")
    print("  START -> decide")
    print("  decide -> [route] -> retrieve_docs -> answer_with_retrieval -> END")
    print("  decide -> [route] -> answer_without_retrieval -> END")


### 12. Test: Query That Needs Retrieval


In [ ]:
# Alice-specific questions should trigger the retrieval path.

print("=" * 60)
print("TEST 1: Alice-specific question (should retrieve)")
print("=" * 60)
result = graph.invoke({
    "query": "What happened when Alice fell down the rabbit hole?",
    "decision": "",
    "retrieved_docs": [],
    "context": "",
    "answer": ""
})
print(f"\nDecision: {result['decision']}")
print(f"Docs retrieved: {len(result['retrieved_docs'])}")
print(f"Answer: {result['answer'][:300]}")


### 13. Test: General Knowledge Query (No Retrieval Needed)


In [ ]:
# A general question should skip retrieval entirely.

print("\n" + "=" * 60)
print("TEST 2: General knowledge question (should NOT retrieve)")
print("=" * 60)
result = graph.invoke({
    "query": "What is the capital of France?",
    "decision": "",
    "retrieved_docs": [],
    "context": "",
    "answer": ""
})
print(f"\nDecision: {result['decision']}")
print(f"Docs retrieved: {len(result['retrieved_docs'])}")
print(f"Answer: {result['answer'][:300]}")


### 14. Test: Edge Case -- Greeting


In [ ]:
print("\n" + "=" * 60)
print("TEST 3: Greeting (should NOT retrieve)")
print("=" * 60)
result = graph.invoke({
    "query": "Hello! How are you doing today?",
    "decision": "",
    "retrieved_docs": [],
    "context": "",
    "answer": ""
})
print(f"\nDecision: {result['decision']}")
print(f"Docs retrieved: {len(result['retrieved_docs'])}")
print(f"Answer: {result['answer'][:300]}")


### 15. Inspect Graph Structure


In [ ]:
print("=== Graph Structure ===")
g = graph.get_graph()
print(f"Nodes: {list(g.nodes.keys())}")
print("Edges:")
for edge in g.edges:
    print(f"  {edge.source} -> {edge.target}")


### Summary
